# 11 Margin Model Testing

This notebook tests different regression models for predicting NFL game margin of victory.

The classification model predicts who wins. The margin model predicts by how many points.

Target variable:

`home_point_diff = home_score - away_score`

A positive predicted margin means the home team is projected to win.  
A negative predicted margin means the away team is projected to win.

In [1]:
# Import packages
import pandas as pd
import numpy as np
import sys

sys.path.append("../src")

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from data_loader import load_game_results
from feature_engineering import create_modeling_dataset
from elo import create_elo_features

In [2]:
# Load game results
game_results = load_game_results("../data/processed/game_results_2018_2025.csv")

game_results.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0


In [3]:
# Create modeling dataset
modeling_data = create_modeling_dataset(game_results)

modeling_data.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,last3_avg_points_scored_diff,last3_avg_points_allowed_diff,last3_avg_point_diff_diff,last3_win_pct_diff,home_strength_of_schedule_before,home_current_opponent_win_pct_before,away_strength_of_schedule_before,away_current_opponent_win_pct_before,strength_of_schedule_diff,current_opponent_win_pct_diff
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Add Elo features
elo_features = create_elo_features(game_results)

elo_cols = [
    "game_id",
    "home_elo_before",
    "away_elo_before",
    "elo_diff",
    "home_elo_with_hfa_diff",
    "elo_home_win_prob"
]

modeling_data = modeling_data.merge(
    elo_features[elo_cols],
    on="game_id",
    how="left"
)

modeling_data.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,home_current_opponent_win_pct_before,away_strength_of_schedule_before,away_current_opponent_win_pct_before,strength_of_schedule_diff,current_opponent_win_pct_diff,home_elo_before,away_elo_before,elo_diff,home_elo_with_hfa_diff,elo_home_win_prob
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,1500.0,1500.0,0.0,55.0,0.578497


In [5]:
# Check target variables
modeling_data[
    [
        "season",
        "week",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_point_diff"
    ]
].head()

,season,week,home_team,away_team,home_score,away_score,home_point_diff
0,2018,1,PHI,ATL,18.0,12.0,6.0
1,2018,1,BAL,BUF,47.0,3.0,44.0
2,2018,1,CLE,PIT,21.0,21.0,0.0
3,2018,1,IND,CIN,23.0,34.0,-11.0
4,2018,1,MIA,TEN,27.0,20.0,7.0


In [6]:
# Choose features and target
features = [
    "avg_points_scored_diff",
    "avg_points_allowed_diff",
    "avg_point_diff_diff",
    "win_pct_diff",
    "last3_avg_points_scored_diff",
    "last3_avg_points_allowed_diff",
    "last3_avg_point_diff_diff",
    "last3_win_pct_diff",
    "elo_diff",
    "home_elo_with_hfa_diff",
    "elo_home_win_prob",
    "strength_of_schedule_diff",
    "current_opponent_win_pct_diff"
]

target = "home_point_diff"

In [7]:
# Train/test split by season
train_data = modeling_data[
    modeling_data["season"].between(2018, 2024)
].copy()

test_data = modeling_data[
    modeling_data["season"] == 2025
].copy()

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 1942
Testing rows: 285


In [8]:
# Check missing values
print("Missing values in X_train:")
print(X_train.isna().sum())

print("\nMissing values in X_test:")
print(X_test.isna().sum())

Missing values in X_train:
avg_points_scored_diff           0
avg_points_allowed_diff          0
avg_point_diff_diff              0
win_pct_diff                     0
last3_avg_points_scored_diff     0
last3_avg_points_allowed_diff    0
last3_avg_point_diff_diff        0
last3_win_pct_diff               0
elo_diff                         0
home_elo_with_hfa_diff           0
elo_home_win_prob                0
strength_of_schedule_diff        0
current_opponent_win_pct_diff    0
dtype: int64

Missing values in X_test:
avg_points_scored_diff           0
avg_points_allowed_diff          0
avg_point_diff_diff              0
win_pct_diff                     0
last3_avg_points_scored_diff     0
last3_avg_points_allowed_diff    0
last3_avg_point_diff_diff        0
last3_win_pct_diff               0
elo_diff                         0
home_elo_with_hfa_diff           0
elo_home_win_prob                0
strength_of_schedule_diff        0
current_opponent_win_pct_diff    0
dtype: int64


In [9]:
# Create regression model
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01, max_iter=10000),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        max_depth=5,
        random_state=42
    ),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        random_state=42
    )
}

In [10]:
# Train and evaluate models
model_results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    model_results.append(
        {
            "model": model_name,
            "mae": mae,
            "rmse": rmse,
            "r2": r2
        }
    )

model_results_df = pd.DataFrame(model_results)

model_results_df

,model,mae,rmse,r2
0,Linear Regression,10.470670,13.105720,0.140081
1,Ridge Regression,10.397598,13.063999,0.145547
2,Lasso Regression,10.374376,13.059392,0.146150
3,Random Forest Regressor,10.275869,12.938784,0.161848
4,Gradient Boosting Regressor,10.350488,13.224541,0.124417


In [11]:
# Sort by MAE
model_results_df = model_results_df.sort_values("mae").reset_index(drop=True)

model_results_df

,model,mae,rmse,r2
0,Random Forest Regressor,10.275869,12.938784,0.161848
1,Gradient Boosting Regressor,10.350488,13.224541,0.124417
2,Lasso Regression,10.374376,13.059392,0.146150
3,Ridge Regression,10.397598,13.063999,0.145547
4,Linear Regression,10.470670,13.105720,0.140081


In [12]:
# Pick best margin model
best_model_name = model_results_df.iloc[0]["model"]
best_mae = model_results_df.iloc[0]["mae"]
best_rmse = model_results_df.iloc[0]["rmse"]
best_r2 = model_results_df.iloc[0]["r2"]

print("Best margin model:", best_model_name)
print(f"Best MAE: {best_mae:.2f} points")
print(f"Best RMSE: {best_rmse:.2f} points")
print(f"Best R²: {best_r2:.3f}")

Best margin model: Random Forest Regressor
Best MAE: 10.28 points
Best RMSE: 12.94 points
Best R²: 0.162


In [13]:
#Train best model again
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

margin_pred = best_model.predict(X_test)

In [14]:
# Create margin results table
margin_results = test_data[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_point_diff"
    ]
].copy()

margin_results["predicted_home_margin"] = margin_pred.round(1)

margin_results["absolute_error"] = (
    margin_results["home_point_diff"] - margin_results["predicted_home_margin"]
).abs().round(1)

margin_results["predicted_margin_text"] = margin_results.apply(
    lambda row: (
        f"{row['home_team']} by {abs(row['predicted_home_margin'])}"
        if row["predicted_home_margin"] >= 0
        else f"{row['away_team']} by {abs(row['predicted_home_margin'])}"
    ),
    axis=1
)

margin_results["actual_margin_text"] = margin_results.apply(
    lambda row: (
        f"{row['home_team']} by {abs(row['home_point_diff'])}"
        if row["home_point_diff"] >= 0
        else f"{row['away_team']} by {abs(row['home_point_diff'])}"
    ),
    axis=1
)

margin_results.head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_point_diff,predicted_home_margin,absolute_error,predicted_margin_text,actual_margin_text
1942,2025,1,2025_01_DAL_PHI,2025-09-04,PHI,DAL,24.0,20.0,4.0,5.8,1.8,PHI by 5.8,PHI by 4.0
1943,2025,1,2025_01_KC_LAC,2025-09-05,LAC,KC,27.0,21.0,6.0,-3.6,9.6,KC by 3.6,LAC by 6.0
1944,2025,1,2025_01_TB_ATL,2025-09-07,ATL,TB,20.0,23.0,-3.0,-3.8,0.8,TB by 3.8,TB by 3.0
1945,2025,1,2025_01_CIN_CLE,2025-09-07,CLE,CIN,16.0,17.0,-1.0,-4.1,3.1,CIN by 4.1,CIN by 1.0
1946,2025,1,2025_01_MIA_IND,2025-09-07,IND,MIA,33.0,8.0,25.0,-1.2,26.2,MIA by 1.2,IND by 25.0
1947,2025,1,2025_01_CAR_JAX,2025-09-07,JAX,CAR,26.0,10.0,16.0,2.8,13.2,JAX by 2.8,JAX by 16.0
1948,2025,1,2025_01_LV_NE,2025-09-07,NE,LV,13.0,20.0,-7.0,0.3,7.3,NE by 0.3,LV by 7.0
1949,2025,1,2025_01_ARI_NO,2025-09-07,NO,ARI,13.0,20.0,-7.0,2.7,9.7,NO by 2.7,ARI by 7.0
1950,2025,1,2025_01_PIT_NYJ,2025-09-07,NYJ,PIT,32.0,34.0,-2.0,-4.1,2.1,PIT by 4.1,PIT by 2.0
1951,2025,1,2025_01_NYG_WAS,2025-09-07,WAS,NYG,21.0,6.0,15.0,6.3,8.7,WAS by 6.3,WAS by 15.0


In [16]:
# View best margin predictions
margin_results.sort_values("absolute_error").head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_point_diff,predicted_home_margin,absolute_error,predicted_margin_text,actual_margin_text
2071,2025,9,2025_09_LAC_TEN,2025-11-02,TEN,LAC,20.0,27.0,-7.0,-7.1,0.1,LAC by 7.1,LAC by 7.0
2225,2025,21,2025_21_LA_SEA,2026-01-25,SEA,LA,31.0,27.0,4.0,3.9,0.1,SEA by 3.9,SEA by 4.0
2166,2025,16,2025_16_LA_SEA,2025-12-18,SEA,LA,38.0,37.0,1.0,0.9,0.1,SEA by 0.9,SEA by 1.0
2003,2025,4,2025_04_GB_DAL,2025-09-28,DAL,GB,40.0,40.0,0.0,0.1,0.1,DAL by 0.1,DAL by 0.0
2101,2025,11,2025_11_SEA_LA,2025-11-16,LA,SEA,21.0,19.0,2.0,2.2,0.2,LA by 2.2,LA by 2.0
2145,2025,14,2025_14_DEN_LV,2025-12-07,LV,DEN,17.0,24.0,-7.0,-7.3,0.3,DEN by 7.3,DEN by 7.0
2156,2025,15,2025_15_BUF_NE,2025-12-14,NE,BUF,31.0,35.0,-4.0,-3.6,0.4,BUF by 3.6,BUF by 4.0
1984,2025,3,2025_03_DEN_LAC,2025-09-21,LAC,DEN,23.0,20.0,3.0,3.4,0.4,LAC by 3.4,LAC by 3.0
2097,2025,11,2025_11_GB_NYG,2025-11-16,NYG,GB,20.0,27.0,-7.0,-6.6,0.4,GB by 6.6,GB by 7.0
1955,2025,1,2025_01_HOU_LA,2025-09-07,LA,HOU,14.0,9.0,5.0,4.5,0.5,LA by 4.5,LA by 5.0


In [17]:
# View worst margin predictions
margin_results.sort_values("absolute_error", ascending=False).head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_point_diff,predicted_home_margin,absolute_error,predicted_margin_text,actual_margin_text
2008,2025,5,2025_05_HOU_BAL,2025-10-05,BAL,HOU,10.0,44.0,-34.0,6.8,40.8,BAL by 6.8,HOU by 34.0
1978,2025,3,2025_03_CIN_MIN,2025-09-21,MIN,CIN,48.0,10.0,38.0,0.2,37.8,MIN by 0.2,MIN by 38.0
1975,2025,3,2025_03_ATL_CAR,2025-09-21,CAR,ATL,30.0,0.0,30.0,-6.6,36.6,ATL by 6.6,CAR by 30.0
2095,2025,11,2025_11_LAC_JAX,2025-11-16,JAX,LAC,35.0,6.0,29.0,-5.7,34.7,LAC by 5.7,JAX by 29.0
2221,2025,20,2025_20_SF_SEA,2026-01-17,SEA,SF,41.0,6.0,35.0,4.7,30.3,SEA by 4.7,SEA by 35.0
2142,2025,14,2025_14_WAS_MIN,2025-12-07,MIN,WAS,31.0,0.0,31.0,1.8,29.2,MIN by 1.8,MIN by 31.0
2050,2025,8,2025_08_MIN_LAC,2025-10-23,LAC,MIN,37.0,10.0,27.0,-1.9,28.9,MIN by 1.9,LAC by 27.0
2005,2025,4,2025_04_CIN_DEN,2025-09-29,DEN,CIN,28.0,3.0,25.0,-3.6,28.6,CIN by 3.6,DEN by 25.0
2038,2025,7,2025_07_MIA_CLE,2025-10-19,CLE,MIA,31.0,6.0,25.0,-2.4,27.4,MIA by 2.4,CLE by 25.0
2122,2025,13,2025_13_CIN_BAL,2025-11-27,BAL,CIN,14.0,32.0,-18.0,9.3,27.3,BAL by 9.3,CIN by 18.0


In [18]:
# Direction accuracy
margin_results["predicted_home_win_from_margin"] = (
    margin_results["predicted_home_margin"] > 0
).astype(int)

margin_results["actual_home_win"] = (
    margin_results["home_point_diff"] > 0
).astype(int)

margin_results["margin_direction_correct"] = (
    margin_results["predicted_home_win_from_margin"] == margin_results["actual_home_win"]
)

direction_accuracy = margin_results["margin_direction_correct"].mean()

print(f"Margin model winner direction accuracy: {direction_accuracy:.2%}")

Margin model winner direction accuracy: 63.16%


In [19]:
# Error by week
error_by_week = (
    margin_results.groupby("week")["absolute_error"]
    .mean()
    .reset_index()
)

error_by_week = error_by_week.rename(
    columns={"absolute_error": "mean_absolute_error"}
)

error_by_week

,week,mean_absolute_error
0,1,6.762500
1,2,9.737500
2,3,13.306250
3,4,9.675000
4,5,12.514286
5,6,8.326667
6,7,10.913333
7,8,15.515385
8,9,8.928571
9,10,9.692857


In [ ]:
# Save margin model results
model_results_df.to_csv(
    "../data/predictions/margin_model_comparison_results.csv",
    index=False
)

margin_results.to_csv(
    "../data/predictions/best_margin_model_2025_predictions.csv",
    index=False
)

print("Saved margin model comparison results.")
print("Saved best margin model 2025 predictions.")

Saved margin model comparison results.
Saved best margin model 2025 predictions.


In [21]:
# Final summary
print("Margin Model Testing Summary")
print("----------------------------")
print("Best model:", best_model_name)
print(f"MAE: {best_mae:.2f} points")
print(f"RMSE: {best_rmse:.2f} points")
print(f"R²: {best_r2:.3f}")
print(f"Winner direction accuracy: {direction_accuracy:.2%}")

Margin Model Testing Summary
----------------------------
Best model: Random Forest Regressor
MAE: 10.28 points
RMSE: 12.94 points
R²: 0.162
Winner direction accuracy: 63.16%


## Summary

This notebook tested several regression models for predicting margin of victory.

The target variable was `home_point_diff`, which measures how many points the home team won or lost by.

The most important metric is MAE, or mean absolute error. MAE tells us how many points the model is off by on average.

The best model from this notebook should be used in `src/predict_upcoming.py` for projected margins.